# Laboratorio 4 - Segmentación de imágenes con técnicas de agrupación
### Caso DermAlpes

**Integrantes:**
- Samuel Molina Luna
- Daniel Pardo Pardo

## Descripción del laboratorio

**DermAlpes** es una empresa de telemedicina que conecta centros de salud de municipios pequeños con dermatólogos de las ciudades principales. En los centros de salud se toman fotografías de lesiones de piel con un dermatoscopio y se envían para que un especialista las revise a distancia. Para agilizar esa revisión, la empresa quiere una herramienta que **delimite automáticamente la zona de la lesión** en cada imagen, de modo que el dermatólogo pueda medir su tamaño y su forma y concentrar su atención en la región relevante.

DermAlpes no cuenta con imágenes etiquetadas de sus propios pacientes, así que como primera aproximación quiere evaluar si es posible segmentar la lesión **sin entrenar un modelo supervisado**, agrupando los píxeles de la imagen según su **color** y su **posición**. Para validar la idea, se construye un método de segmentación sobre un conjunto de imágenes públicas que sí incluye máscaras de referencia elaboradas por dermatólogos, y así se mide qué tan bien funciona, dónde falla y por qué.

La herramienta **no realiza diagnósticos**: su único propósito es delimitar la lesión como apoyo para el especialista. Aun así, la empresa quiere entender las limitaciones del método y los riesgos de usarlo con pacientes cuyas características difieran de las de las imágenes con que fue construido.

## Conjunto de datos

Para el conjunto de datos se utiliza una selección de 20 imágenes dermatoscópicas de ISIC 2016. Cada imagen viene acompañada de una máscara de referencia que identifica los píxeles correspondientes a la lesión. Estas máscaras no se usan para agrupar los píxeles sino que solo sirven para comparar y evaluar los resultados del método.

Las imágenes se dividen en dos conjuntos:

| Conjunto | Cantidad | Uso |
|---|---|---|
| **Ajuste** | 15 imágenes | Explorar y seleccionar la configuración del método comparando las máscaras estimadas con las de referencia |
| **Prueba** | 5 imágenes | Evaluar el desempeño final sin realizar modificaciones adicionales |

| Imágenes de ajuste | | | Imágenes de prueba |
|---|---|---|---|
| ISIC_0000085 | ISIC_0000114 | ISIC_0000150 | ISIC_0000087 |
| ISIC_0000086 | ISIC_0000116 | ISIC_0000153 | ISIC_0000105 |
| ISIC_0000089 | ISIC_0000122 | ISIC_0000156 | ISIC_0000110 |
| ISIC_0000094 | ISIC_0000133 | ISIC_0000157 | ISIC_0000123 |
| ISIC_0000108 | ISIC_0000137 | ISIC_0000112 | ISIC_0000128 |

**Preparación de las máscaras.** Las máscaras originales contienen los valores 0 (piel o fondo) y 255 (lesión), que se convierten a 0 y 1. Si se reduce el tamaño de las imágenes, las máscaras se redimensionan con interpolación por vecino más cercano para conservar sus valores binarios.

**Fuente.** International Skin Imaging Collaboration (ISIC), *ISIC 2016 Challenge - Task 1: Lesion Segmentation*. Disponible en: https://challenge.isic-archive.com/landing/2016/37/

Referencia del desafío: Gutman, D., Codella, N. C. F., Celebi, E., et al. (2016). *Skin Lesion Analysis toward Melanoma Detection: A Challenge at the International Symposium on Biomedical Imaging (ISBI) 2016, hosted by the International Skin Imaging Collaboration (ISIC)*. arXiv:1605.01397.

# Importación de librerías

Se importan las librerías que se usan a lo largo del laboratorio: NumPy y pandas para los datos y las tablas de resultados, Pillow para leer las imágenes, Matplotlib y Seaborn para las visualizaciones, y Scikit-learn para el pipeline, K-medias y las métricas (silueta, Dice a través de `f1_score`, V-measure, precisión y recall).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import (silhouette_score, f1_score, v_measure_score,
                             precision_score, recall_score)